# Download Liars' Bench probe activations on Colab → save to Google Drive

This notebook **drives** the white-box Liars' Bench probe evaluation
(`deception_detection/scripts/experiment.py`) from Colab, running the benchmark's scenario configs
**unchanged**, while **all code stays in GitHub** and **all outputs go to Google Drive**.

```
VS Code (edit) --git push--> GitHub <--git pull-- Colab (run) --> Google Drive (outputs)
```

**How the workflow fits together**
- Edit code in **VS Code** on your laptop, then `git push`.
- In Colab, re-run the *Clone / update repo* cell (`git pull`) to get your changes, then re-run.
- The model runs **remotely on NDIF** (`model.trace(..., remote=True)`), so Colab only orchestrates
  and downloads the activations — you do **not** need a big GPU here.
- Activations + results are written to **Drive**, and runs are **resumable**: activations are cached
  **per batch** with atomic writes, so on a Colab disconnect you reconnect and re-run — completed
  batches/scenarios load from the Drive cache and only the remainder is re-downloaded from NDIF.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.** (Even though compute is remote, a CUDA runtime avoids
   an error in the entry script's memory-logging line. A free T4 is plenty.)
2. Add your keys under **🔑 Secrets** (left sidebar): `NDIF_API_KEY` (from https://login.ndif.us) and
   `HF_TOKEN`. Toggle *Notebook access* on for each.
3. **Model availability.** These are the real benchmark configs, so each carries its own model. Only
   the base-model scenario **`XXXX-9`** (Llama-70B-3.3) runs on stock NDIF. The LoRA/Gemma
   model-organism scenarios (`XXXX-15/16/17/18/19`) need those weights hosted on NDIF (or run
   locally) — otherwise they report `FAILED` in the summary. The notebook attempts all and keeps going.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Edit these to match your repo, branch, models and config file.

In [ ]:
# --- Repo -------------------------------------------------------------------
REPO_URL = 'https://github.com/zami456/expLB.git'
BRANCH   = 'claude/colab-activation-download-drive'  # branch you develop on in VS Code
REPO_DIR = '/content/expLB'
# All extraction commands run from the vendored probes package:
WORKDIR  = REPO_DIR + '/src/probes/deception-detection'

# --- Google Drive output locations -----------------------------------------
DRIVE_BASE     = '/content/drive/MyDrive/expLB'
OUT            = DRIVE_BASE + '/results'             # experiment folders (detector.pt, *_scores.json)
ACTS_CACHE_DIR = DRIVE_BASE + '/activations_cache'   # cached NDIF activations -> enables resume

# --- What to run: the Liars' Bench scenario configs -------------------------
# Each XXXX-* config IS a Liars' Bench scenario (README: "each YAML targets a Liars' Bench
# scenario"). They run UNCHANGED -- own model, LoRA, datasets, control -- we only redirect outputs
# to Drive. Do NOT edit datasets/model here; that would diverge from the benchmark paper + code.
CFG_DIR = 'deception_detection/scripts/configs'
CONFIGS = [
    f'{CFG_DIR}/XXXX-9.yaml',    # harm-refusal + instructed-dishonesty | Llama-70B-3.3 base -> NDIF-remote OK
    f'{CFG_DIR}/XXXX-15.yaml',   # convincing-game                      | Llama-70B-3.1 + LoRA
    f'{CFG_DIR}/XXXX-16.yaml',   # gender-secret                        | Llama-70B-3.3 + LoRA
    f'{CFG_DIR}/XXXX-17.yaml',   # sleeper-agents (gender)              | Gemma-27B + LoRA
    f'{CFG_DIR}/XXXX-18.yaml',   # sleeper-agents (greeting)            | Gemma-27B + LoRA
    f'{CFG_DIR}/XXXX-19.yaml',   # sleeper-agents (time)                | Gemma-27B + LoRA
]
# NOTE: only XXXX-9 uses a plain base model (loads via nnsight LanguageModel) and runs on stock
# NDIF. XXXX-15/16/17/18/19 use fine-tuned / LoRA model organisms (and Gemma loads locally), so
# they require those weights to be available (hosted on NDIF, or run locally) or they fail at load.
# The run loop below attempts every config and reports per-scenario status instead of aborting.

import os
os.makedirs(OUT, exist_ok=True)
os.makedirs(ACTS_CACHE_DIR, exist_ok=True)
print('Outputs   ->', OUT)
print('Acts cache->', ACTS_CACHE_DIR)
print(f'{len(CONFIGS)} Liars\' Bench scenario configs queued')

## 3. Clone / update the repo

Run this cell every time you push new code from VS Code — it clones on the first run and `git pull`s
afterwards. For a **private** repo, replace `REPO_URL` with a token URL
(`https://<user>:<token>@github.com/zami456/expLB.git`) or use `gh auth`.

In [ ]:
import os, subprocess

if not os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

print(subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)

## 4. Install dependencies

Installs the `deception_detection` package from the vendored probes dir — this pulls in `nnsight`
and everything else the extraction needs. Takes a few minutes; you only need to re-run it when
dependencies change.

In [ ]:
!pip install -q -e {WORKDIR}

## 5. Secrets & remote (NDIF) configuration

Reads `NDIF_API_KEY` / `HF_TOKEN` from Colab Secrets, sets them in the environment, points the
activation cache at Drive (`ACTS_CACHE_DIR`), and registers the NDIF key with nnsight (persisted to
nnsight's config so the subprocess runs in the next cell are authenticated).

In [ ]:
import os
from google.colab import userdata

os.environ['NDIF_API_KEY']   = userdata.get('NDIF_API_KEY')
os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['ACTS_CACHE_DIR'] = ACTS_CACHE_DIR  # read by Activations.from_model (get_acts_cache_dir)

import nnsight
nnsight.CONFIG.set_default_api_key(os.environ['NDIF_API_KEY'])
print('NDIF key + HF token configured; cache ->', os.environ['ACTS_CACHE_DIR'])

## 6. Run extraction (loops over the Liars' Bench scenario configs)

Runs each `XXXX-*` scenario config **unchanged** (its own model / LoRA / datasets / control), writing
to Drive with a stable, timestamp-free name (the config stem, e.g. `XXXX-9`). The loop attempts every
config and prints a per-scenario **status summary** at the end, so a config NDIF can't serve doesn't
stop the others. **tqdm download bars stream into this cell's output.**

**Reminder:** only `XXXX-9` (base Llama-70B) runs on stock NDIF. The LoRA/Gemma scenarios
(`XXXX-15/16/17/18/19`) need their model-organism weights available or they'll report `FAILED` — run
those locally (where the `Saving`-branch cache applies).

**Resume:** reconnect after a disconnect and re-run this cell — two levels resume: completed per-batch
`.pt` files load from `ACTS_CACHE_DIR` (no NDIF re-download), and finished `detector.pt` / `scores.json`
in the stable-named results folder are skipped.

In [ ]:
import subprocess
from pathlib import Path

results = {}
for cfg in CONFIGS:
    name = Path(cfg).stem  # stable, timestamp-free -> results folder resumes across re-runs
    print(f"\n{'='*72}\n=== Liars' Bench scenario: {name}\n{'='*72}", flush=True)
    cmd = [
        'python', 'deception_detection/scripts/experiment.py', 'run',
        '--config-file', cfg,
        f'--name={name}',
        f'--folder={OUT}',
    ]
    # No check=True: attempt every scenario and record status, so one failing config
    # (e.g. a LoRA/Gemma organism NDIF can't host) doesn't stop the rest.
    rc = subprocess.run(cmd, cwd=WORKDIR).returncode
    results[name] = 'OK' if rc == 0 else f'FAILED (exit {rc})'

print(f"\n{'='*72}\nSummary (outputs on Drive at {OUT})\n{'='*72}")
for name, status in results.items():
    print(f"  {name:<10} {status}")

## Notes

- **Watch progress**: tqdm bars above + browse the Drive folders (`results/`, `activations_cache/`).
- **Change code**: edit in VS Code → `git push` → re-run cell **3** (`git pull`) → re-run cell **6**.
- **The configs are the benchmark**: `XXXX-*.yaml` are the Liars' Bench scenarios, run unchanged. To
  add/remove scenarios, edit the `CONFIGS` list — not the datasets inside a config.
- **Resume granularity**: the cache unit is a *batch* (`BATCH_SIZE_ACTS`, default 32); a mid-batch
  interruption re-downloads only that batch (atomic writes prevent corrupt partial caches).
- **Colab idle**: sessions disconnect on idle (~90 min) or at runtime caps — expected; the resume
  design makes it harmless. Don't use anti-idle scripts (against Colab ToS).